## Consultas - Saldos OC

#### 1.Carregar tabela da camada silver

In [1]:
import sys
sys.path.append("/app")


from utils import create_spark_session, load_config, save_table
from pyspark.sql import functions as F
from delta.tables import DeltaTable


spark = create_spark_session("saldos_oc")
config = load_config()

# ler tabelas silver
df_compras_ordens_itens = spark.read.format("delta").load(
    f"data/silver/compras_ordens_itens"
)
df_rel_compras_ordens_referenciadas_saldos = spark.read.format("delta").load(
    f"data/silver/rel_compras_ordens_referenciadas_saldos"
)
df_produtos = spark.read.format("delta").load(
    f"data/silver/produtos"
)
df_pessoas = spark.read.format("delta").load(
    f"data/silver/pessoas"
)
df_compras_ordens = spark.read.format("delta").load(
    f"data/silver/compras_ordens"
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/12 01:51:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/12 01:51:20 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


#### 2.Executar transformação

In [2]:

# transformação

df_compras_ordens_itens.createOrReplaceTempView("compras_ordens_itens")
df_rel_compras_ordens_referenciadas_saldos.createOrReplaceTempView("rel_compras_ordens_referenciadas_saldos")
df_produtos.createOrReplaceTempView("produtos")
df_pessoas.createOrReplaceTempView("pessoas")
df_compras_ordens.createOrReplaceTempView("compras_ordens")

df_gold = spark.sql(
    """
    SELECT
        coi.ordem_compra_id AS ordem_compra_id, 
        OC.status as status, 
        coi.produto_id AS produto_id, 
        P.codigo_identificacao_interno AS codigo_identificacao_interno, 
        P.NOME AS NOME, P.FORNECEDOR_ID AS FORNECEDOR_ID, 
        FORNECEDOR.razao_social AS razao_social,
        date_format(coi.prazo_entrega, 'dd/MM/yyyy') AS prazo_entrega,
        coi.quantidade AS quantidade, 
        sld.saldo AS saldo, 
        coi.valor_unitario AS valor_unitario, 
        coi.valor_total AS valor_total

    FROM compras_ordens_itens coi
    LEFT JOIN rel_compras_ordens_referenciadas_saldos SLD  
        ON coi.ordem_compra_id = sld.ordem_compra_id 
    AND coi.produto_id = sld.produto_id 
    AND SLD.id_item = COI.id
    LEFT JOIN PRODUTOS P 
        ON COI.produto_id = P.id
    LEFT JOIN pessoas FORNECEDOR 
        ON FORNECEDOR.id = P.fornecedor_id
    LEFT JOIN compras_ordens OC 
        ON COI.ordem_compra_id =  OC.id
                            
    WHERE 
        --sld.produto_id = 101
        --coi.ordem_compra_id IN(584,583,582) 
        --AND sld.empresa_id = 1  
        sld.SALDO <> 0 
    --ORDER BY 
        --coi.produto_id, 
        --year(coi.prazo_entrega), 
        --month(coi.prazo_entrega), 
        --day(coi.prazo_entrega); 
    """
)

# ordenação
df_gold = df_gold.orderBy(
    "produto_id",
    F.year("prazo_entrega"),
    F.month("prazo_entrega"),
    F.day("prazo_entrega")
)

#### 3.Armazenar dados na camada gold

In [3]:
# salvar gold

path = "data/gold/saldos_oc"

df_gold.write.format("delta").mode("overwrite").save(path)